<h1>Tarea Ramdom Forest<h1>

<p>Escogi un Dataset de los ataques de los tiburones que lo cogi de aqui: https://www.kaggle.com/datasets/gauravkumar2525/shark-attacks?resource=download <p>

<p>Lo que hacemos aqui es que con el Ramdom Forest Clasificador vamos a predecir si un ataque fue fatal y con el Ramdom Forest Regresion vamos a predecir la edad de a victima. Tambien vamos a evaluar los modelos con precision y error absoluto medio.<p>

<h1>Random Forest Classifier<h1>

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier, VotingRegressor, StackingClassifier, StackingRegressor, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import accuracy_score, mean_absolute_error

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

<p>Ahora vamos a cargar los datos<p>

In [2]:
# Cargar dataset
df = pd.read_csv("global_shark_attacks.csv")
df.head()

,date,year,type,country,area,location,activity,name,sex,age,fatal_y_n,time,species
0,2023-05-13,2023.0,Unprovoked,AUSTRALIA,South Australia,Elliston,Surfing,Simon Baccanello,M,46,Y,10h10,White shark
1,2023-04-29,2023.0,Unprovoked,AUSTRALIA,Western Australia,"Yallingup, Busselton",Swimming,male,M,NaN,N,11h20,1m shark
2,2022-10-07,2022.0,Unprovoked,AUSTRALIA,Western Australia,Port Hedland,Spearfishing,Robbie Peck,M,38,N,11h30,Bull shark
3,2021-10-04,2021.0,Unprovoked,USA,Florida,"Fort Pierce State Park, St. Lucie County",Surfing,Truman Van Patrick,M,25.0,N,NaN,NaN
4,2021-10-03,2021.0,Unprovoked,USA,Florida,"Jensen Beach, Martin County",Swimming,male,M,NaN,N,12h00,NaN


<p>Ahora vamos a eliminar las filas con valores nulos en la variable objetivo<p>

In [3]:
# --- CLASIFICACIÓN (Fatal Y/N) ---

# Eliminar filas con valores nulos en la variable objetivo
df_class = df.dropna(subset=["fatal_y_n"])
df.head()

,date,year,type,country,area,location,activity,name,sex,age,fatal_y_n,time,species
0,2023-05-13,2023.0,Unprovoked,AUSTRALIA,South Australia,Elliston,Surfing,Simon Baccanello,M,46,Y,10h10,White shark
1,2023-04-29,2023.0,Unprovoked,AUSTRALIA,Western Australia,"Yallingup, Busselton",Swimming,male,M,NaN,N,11h20,1m shark
2,2022-10-07,2022.0,Unprovoked,AUSTRALIA,Western Australia,Port Hedland,Spearfishing,Robbie Peck,M,38,N,11h30,Bull shark
3,2021-10-04,2021.0,Unprovoked,USA,Florida,"Fort Pierce State Park, St. Lucie County",Surfing,Truman Van Patrick,M,25.0,N,NaN,NaN
4,2021-10-03,2021.0,Unprovoked,USA,Florida,"Jensen Beach, Martin County",Swimming,male,M,NaN,N,12h00,NaN


<p> Ahora vamos a seleccionar las características mas relevantes
Elegimos columnas que podrían influir en el resultado:

    Año del ataque (year)
    País (country)
    Sexo de la víctima (sex)
    Actividad (activity) 
<p>

In [4]:
# Seleccionar características relevantes
features = ["year", "country", "sex", "activity"]
df_class = df_class[features + ["fatal_y_n"]].dropna()
df_class.head()

,year,country,sex,activity,fatal_y_n
0,2023.0,AUSTRALIA,M,Surfing,Y
1,2023.0,AUSTRALIA,M,Swimming,N
2,2022.0,AUSTRALIA,M,Spearfishing,N
3,2021.0,USA,M,Surfing,N
4,2021.0,USA,M,Swimming,N


<p> Ahora vamos a convertir las variables categóricas a números
Por ejemplo, country podría convertirse en:

    Australia → 0
    USA → 1
    South Africa → 2
<p>

In [5]:
# Codificar variables categóricas
encoder = LabelEncoder()
df_class["country"] = encoder.fit_transform(df_class["country"])
df_class["sex"] = encoder.fit_transform(df_class["sex"])
df_class["activity"] = encoder.fit_transform(df_class["activity"])
df_class["fatal_y_n"] = encoder.fit_transform(df_class["fatal_y_n"])
df_class.head()

,year,country,sex,activity,fatal_y_n
0,2023.0,10,2,1007,5
1,2023.0,10,2,1042,2
2,2022.0,10,2,901,2
3,2021.0,188,2,1007,2
4,2021.0,188,2,1042,2


<p>Ahora vamos a dividir datos en entrenamiento y prueba<p>
<p>Se separan los datos en 80% para entrenamiento y 20% para prueba.
<p>

In [6]:
# Separar datos
X_train, X_test, y_train, y_test = train_test_split(df_class[features], df_class["fatal_y_n"], test_size=0.2, random_state=42)

<p>Ahora vamos a entrenar el modelo de clasificación<p>
<p>Creamos un Random Forest Classifier con 100 árboles y lo entrenamos.<p>

In [7]:
# Entrenar modelo de clasificación
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

<p>Ahora vamos a evaluar el modelo<p>
<p>Se predicen los valores de prueba y calculamos la precisión del modelo.<p>

In [8]:
# Evaluar
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión en clasificación: {accuracy:.4f}")


Precisión en clasificación: 0.7165


<h1>Random Forest Regressor<h1>

<p>Ahora vamos a eliminar valores nulos en la edad<p>
<p>Se eliminan registros sin edad.<p>

In [9]:
# --- REGRESIÓN (Edad) ---

# Eliminar filas con valores nulos en la variable objetivo
df_reg = df.dropna(subset=["age"])
df.head()

,date,year,type,country,area,location,activity,name,sex,age,fatal_y_n,time,species
0,2023-05-13,2023.0,Unprovoked,AUSTRALIA,South Australia,Elliston,Surfing,Simon Baccanello,M,46,Y,10h10,White shark
1,2023-04-29,2023.0,Unprovoked,AUSTRALIA,Western Australia,"Yallingup, Busselton",Swimming,male,M,NaN,N,11h20,1m shark
2,2022-10-07,2022.0,Unprovoked,AUSTRALIA,Western Australia,Port Hedland,Spearfishing,Robbie Peck,M,38,N,11h30,Bull shark
3,2021-10-04,2021.0,Unprovoked,USA,Florida,"Fort Pierce State Park, St. Lucie County",Surfing,Truman Van Patrick,M,25.0,N,NaN,NaN
4,2021-10-03,2021.0,Unprovoked,USA,Florida,"Jensen Beach, Martin County",Swimming,male,M,NaN,N,12h00,NaN


<p>Vamos a convertir age a número<p>
<p>Algunas edades pueden tener texto en vez de números ("Unknown", "Young"). Esta línea los convierte a NaN.<p>

In [10]:
# Convertir edad a numérico
df_reg.loc[:, "age"] = pd.to_numeric(df_reg["age"], errors="coerce")
df.head()


,date,year,type,country,area,location,activity,name,sex,age,fatal_y_n,time,species
0,2023-05-13,2023.0,Unprovoked,AUSTRALIA,South Australia,Elliston,Surfing,Simon Baccanello,M,46,Y,10h10,White shark
1,2023-04-29,2023.0,Unprovoked,AUSTRALIA,Western Australia,"Yallingup, Busselton",Swimming,male,M,NaN,N,11h20,1m shark
2,2022-10-07,2022.0,Unprovoked,AUSTRALIA,Western Australia,Port Hedland,Spearfishing,Robbie Peck,M,38,N,11h30,Bull shark
3,2021-10-04,2021.0,Unprovoked,USA,Florida,"Fort Pierce State Park, St. Lucie County",Surfing,Truman Van Patrick,M,25.0,N,NaN,NaN
4,2021-10-03,2021.0,Unprovoked,USA,Florida,"Jensen Beach, Martin County",Swimming,male,M,NaN,N,12h00,NaN


<p>Ahora vamos a seleccionar características mas relevantes<p>
<p>Nos quedamos con las columnas útiles y eliminamos filas con datos faltantes.<p>

In [11]:

# Seleccionar características relevantes
df_reg = df_reg[features + ["age"]].dropna()
df_reg.head() 

,year,country,sex,activity,age
0,2023.0,AUSTRALIA,M,Surfing,46.0
2,2022.0,AUSTRALIA,M,Spearfishing,38.0
3,2021.0,USA,M,Surfing,25.0
5,2021.0,USA,M,Swimming,26.0
6,2021.0,USA,M,Surfing,14.0


<p>Ahora vamos a codificar las variables categóricas<p>
<p>Transformamos los valores de texto en números.<p>

In [12]:
# Codificar variables categóricas
df_reg["country"] = encoder.fit_transform(df_reg["country"])
df_reg["sex"] = encoder.fit_transform(df_reg["sex"])
df_reg["activity"] = encoder.fit_transform(df_reg["activity"])
df_reg.head()

,year,country,sex,activity,age
0,2023.0,5,1,525,46.0
2,2022.0,5,1,461,38.0
3,2021.0,130,1,525,25.0
5,2021.0,130,1,554,26.0
6,2021.0,130,1,525,14.0


<p>Ahora vamos a dividir datos en entrenamiento y prueba<p>
<p>Separa el dataset en entrenamiento (80%) y prueba (20%).<p>

In [13]:
# Separar datos
X_train, X_test, y_train, y_test = train_test_split(df_reg[features], df_reg["age"], test_size=0.2, random_state=42)

<p>Ahora vamos a entrenar el modelo de regresión<p>
<p>Entrenamos el modelo RandomForestRegressor.<p>

In [14]:
# Entrenar modelo de regresión
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_train, y_train)


RandomForestRegressor(random_state=42)

<p>Ahora vamos a evaluar el modelo<p>
<p>Calculamos el Error Absoluto Medio (MAE), que mide cuántos años en promedio se equivoca el modelo.<p>

In [15]:
# Evaluar
y_pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Error absoluto medio en regresión: {mae:.2f} años")

Error absoluto medio en regresión: 11.78 años


In [21]:
# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=[2, 5])

# Crear el gráfico de la matriz de confusión
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["No Fatal", "Fatal"], yticklabels=["No Fatal", "Fatal"])
plt.title("Matriz de Confusión - Clasificación (Fatal Y/N)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()


NameError: name 'y_pred_class' is not defined

<p>Prueba los "ensemble methods" : comienza por los de Votación (VotingClassifier, VotingRegressor) y continúa con los demás (stacking y boosting: XGBoost)<p>

In [13]:
# Cargar dataset
df = pd.read_csv("global_shark_attacks.csv")

# --- CLASIFICACIÓN (Fatal Y/N) ---
df_class = df.dropna(subset=["fatal_y_n"])
features = ["year", "country", "sex", "activity"]
df_class = df_class[features + ["fatal_y_n"]].dropna().copy()

encoder = LabelEncoder()
df_class.loc[:, "country"] = encoder.fit_transform(df_class["country"].astype(str))
df_class.loc[:, "sex"] = encoder.fit_transform(df_class["sex"].astype(str))
df_class.loc[:, "activity"] = encoder.fit_transform(df_class["activity"].astype(str))
df_class.loc[:, "fatal_y_n"] = encoder.fit_transform(df_class["fatal_y_n"].astype(str))

# Filtrar clases con al menos 5 ejemplos
df_class = df_class[df_class.groupby("fatal_y_n")["fatal_y_n"].transform("count") >= 5]

X_train, X_test, y_train, y_test = train_test_split(df_class[features], df_class["fatal_y_n"].astype(int), test_size=0.2, random_state=42, stratify=df_class["fatal_y_n"].astype(int))

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Voting Classifier
clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
clf2 = SVC(probability=True, random_state=42)
clf3 = LogisticRegression(max_iter=5000, random_state=42)
voting_clf = VotingClassifier(estimators=[('rf', clf1), ('svm', clf2), ('lr', clf3)], voting='soft')
voting_clf.fit(X_train, y_train)
y_pred = voting_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión en Voting Classifier: {accuracy:.4f}")

# Stacking Classifier
stacking_clf = StackingClassifier(estimators=[('rf', clf1), ('svm', clf2), ('lr', clf3)], final_estimator=LogisticRegression(max_iter=5000))
stacking_clf.fit(X_train, y_train)
y_pred = stacking_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión en Stacking Classifier: {accuracy:.4f}")

# --- REGRESIÓN (Edad) ---
df_reg = df.dropna(subset=["age"]).copy()
df_reg.loc[:, "age"] = pd.to_numeric(df_reg["age"], errors="coerce")
df_reg = df_reg[features + ["age"]].dropna()

df_reg.loc[:, "country"] = encoder.fit_transform(df_reg["country"].astype(str))
df_reg.loc[:, "sex"] = encoder.fit_transform(df_reg["sex"].astype(str))
df_reg.loc[:, "activity"] = encoder.fit_transform(df_reg["activity"].astype(str))

X_train, X_test, y_train, y_test = train_test_split(df_reg[features], df_reg["age"].astype(float), test_size=0.2, random_state=42)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Voting Regressor
reg1 = RandomForestRegressor(n_estimators=100, random_state=42)
reg2 = GradientBoostingRegressor(n_estimators=100, random_state=42)
reg3 = xgb.XGBRegressor(n_estimators=100, random_state=42)
voting_reg = VotingRegressor(estimators=[('rf', reg1), ('gbr', reg2), ('xgb', reg3)])
voting_reg.fit(X_train, y_train)
y_pred = voting_reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Error absoluto medio en Voting Regressor: {mae:.4f} años")

# Stacking Regressor
stacking_reg = StackingRegressor(estimators=[('rf', reg1), ('gbr', reg2), ('xgb', reg3)], final_estimator=GradientBoostingRegressor())
stacking_reg.fit(X_train, y_train)
y_pred = stacking_reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Error absoluto medio en Stacking Regressor: {mae:.4f} años")


Precisión en Voting Classifier: 0.7535
Precisión en Stacking Classifier: 0.7674
Error absoluto medio en Voting Regressor: 11.3429 años
Error absoluto medio en Stacking Regressor: 11.3523 años


<p>Me funciona mejor el Stacking porque me da 0,76 mas que los otros <p>